In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
import joblib

In [4]:
df = pd.read_csv("03_Clustering_Marketing.csv")
print(df.head())

   gradyear gender     age  NumberOffriends  basketball  football  soccer  \
0      2007    NaN     NaN                0           0         0       0   
1      2007      F   17.41               49           0         0       1   
2      2007      F  17.511               41           0         0       0   
3      2006      F     NaN               36           0         0       0   
4      2008      F  16.657                1           0         0       0   

   softball  volleyball  swimming  ...  blonde  mall  shopping  clothes  \
0         0           0         0  ...       0     0         0        0   
1         0           0         1  ...       0     0         0        0   
2         0           0         0  ...       0     1         0        0   
3         0           0         0  ...       0     0         0        0   
4         0           0         1  ...       0     0         0        3   

   hollister  abercrombie  die  death  drunk  drugs  
0          0            0    0  

In [6]:
# Basic EDA
print(df.info())
print(df.describe())

# missing values
print(df.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 40 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   gradyear         15000 non-null  int64
 1   gender           13663 non-null  str  
 2   age              12504 non-null  str  
 3   NumberOffriends  15000 non-null  int64
 4   basketball       15000 non-null  int64
 5   football         15000 non-null  int64
 6   soccer           15000 non-null  int64
 7   softball         15000 non-null  int64
 8   volleyball       15000 non-null  int64
 9   swimming         15000 non-null  int64
 10  cheerleading     15000 non-null  int64
 11  baseball         15000 non-null  int64
 12  tennis           15000 non-null  int64
 13  sports           15000 non-null  int64
 14  cute             15000 non-null  int64
 15  sex              15000 non-null  int64
 16  sexy             15000 non-null  int64
 17  hot              15000 non-null  int64
 18  kissed           

In [11]:
# handling missing values
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')


for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna('Unknown')
    else:
        df[col] = df[col].fillna(df[col].median())

In [15]:
print(df.isnull().sum())

gradyear               0
gender             15000
age                    0
NumberOffriends        0
basketball             0
football               0
soccer                 0
softball               0
volleyball             0
swimming               0
cheerleading           0
baseball               0
tennis                 0
sports                 0
cute                   0
sex                    0
sexy                   0
hot                    0
kissed                 0
dance                  0
band                   0
marching               0
music                  0
rock                   0
god                    0
church                 0
jesus                  0
bible                  0
hair                   0
dress                  0
blonde                 0
mall                   0
shopping               0
clothes                0
hollister              0
abercrombie            0
die                    0
death                  0
drunk                  0
drugs                  0


In [16]:

df = df.fillna(df.median(numeric_only=True))

# If still any NaN (categorical cases)
df = df.fillna(0)

In [17]:
# Label encoding
le_dict = {}

for col in df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    le_dict[col] = le

In [18]:
# Feature scaling
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df)

In [26]:
# save feature names
feature_names = df.columns.tolist()

In [20]:
# Trying clustering models
# KMeans

kmeans = KMeans(n_clusters=2, random_state=42)
kmeans_labels = kmeans.fit_predict(scaled_data)

kmeans_score = silhouette_score(scaled_data, kmeans_labels)
print("KMeans Silhouette Score:", kmeans_score)

KMeans Silhouette Score: 0.39071695962349745


In [21]:
# agglomerative
agg = AgglomerativeClustering(n_clusters=4)
agg_labels = agg.fit_predict(scaled_data)

agg_score = silhouette_score(scaled_data, agg_labels)
print("Agglomerative Score:", agg_score)

Agglomerative Score: 0.3674798747058095


In [22]:
# DBSCAN

db = DBSCAN(eps=2.5, min_samples=5)
db_labels = db.fit_predict(scaled_data)

# Remove noise points (-1)
mask = db_labels != -1

if len(set(db_labels[mask])) > 1:
    db_score = silhouette_score(scaled_data[mask], db_labels[mask])
else:
    db_score = -1

print("DBSCAN Score:", db_score)

DBSCAN Score: 0.15321751065822237


In [23]:
# Compare models
scores = {
    "KMeans": kmeans_score,
    "Agglomerative": agg_score,
    "DBSCAN": db_score
}

print(scores)


{'KMeans': 0.39071695962349745, 'Agglomerative': 0.3674798747058095, 'DBSCAN': 0.15321751065822237}


In [27]:
# saving model

joblib.dump(kmeans, "kmeans_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(le_dict, "encoders.pkl")
joblib.dump(feature_names, "features.pkl")

['features.pkl']